# Module 4 Code Exercise

In [8]:
import pandas as pd
import numpy as np
from glob import glob
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from urllib import request

%matplotlib inline

OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
epub_dir = 'epubs'

In [ ]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('tagsets')

In [62]:
books = {
    'middlemarch': {
        'url': 'http://www.gutenberg.org/files/145/145-0.txt',
        'title': 'Middlemarch by George Eliot',
        'book_id': 145
    },
    'mill_on_floss': {
        'url': 'http://www.gutenberg.org/files/6688/6688-0.txt',
        'title': 'The Mill on the Floss by George Eliot',
        'book_id': 6688
    },
    'adam_bede': {
        'url': 'http://www.gutenberg.org/files/507/507-0.txt',
        'title': 'Adam Bede by George Eliot',
        'book_id': 507
    }
}

raw_texts = {}
for key, info in books.items():
    response = request.urlopen(info['url'])
    raw = response.read().decode('utf-8-sig')
    raw_texts[key] = raw
    print(f"Downloaded {info['title']}: {len(raw)} characters")

Downloaded Middlemarch by George Eliot: 1813565 characters
Downloaded The Mill on the Floss by George Eliot: 1182417 characters
Downloaded Adam Bede by George Eliot: 1195779 characters


In [63]:
LIB = pd.DataFrame([
    {
        'book_id': info['book_id'],
        'title': info['title'],
        'source_path': info['url']
    }
    for key, info in books.items()
])

LIB = LIB.set_index('book_id')
print(LIB)

                                         title  \
book_id                                          
145                Middlemarch by George Eliot   
6688     The Mill on the Floss by George Eliot   
507                  Adam Bede by George Eliot   

                                            source_path  
book_id                                                  
145        http://www.gutenberg.org/files/145/145-0.txt  
6688     http://www.gutenberg.org/files/6688/6688-0.txt  
507        http://www.gutenberg.org/files/507/507-0.txt  


In [64]:
lines = raw_texts['middlemarch'].split('\n')
for i, line in enumerate(lines[0:100]):
    print(f"{i}: {line}")

0: *** START OF THE PROJECT GUTENBERG EBOOK 145 ***
1: 
2: 
3: 
4: 
5: Middlemarch
6: 
7: George Eliot
8: 
9: New York and Boston
10: H. M. Caldwell Company Publishers
11: 
12: To my dear Husband, George Henry Lewes,
13: in this nineteenth year of our blessed union.
14: 
15: 
16: Contents
17: 
18:  PRELUDE.
19: 
20:  BOOK I. MISS BROOKE.
21:  CHAPTER I.
22:  CHAPTER II.
23:  CHAPTER III.
24:  CHAPTER IV.
25:  CHAPTER V.
26:  CHAPTER VI.
27:  CHAPTER VII.
28:  CHAPTER VIII.
29:  CHAPTER IX.
30:  CHAPTER X.
31:  CHAPTER XI.
32:  CHAPTER XII.
33: 
34:  BOOK II. OLD AND YOUNG.
35:  CHAPTER XIII.
36:  CHAPTER XIV.
37:  CHAPTER XV.
38:  CHAPTER XVI.
39:  CHAPTER XVII.
40:  CHAPTER XVIII.
41:  CHAPTER XIX.
42:  CHAPTER XX.
43:  CHAPTER XXI.
44:  CHAPTER XXII.
45: 
46:  BOOK III. WAITING FOR DEATH.
47:  CHAPTER XXIII.
48:  CHAPTER XXIV.
49:  CHAPTER XXV.
50:  CHAPTER XXVI.
51:  CHAPTER XXVII.
52:  CHAPTER XXVIII.
53:  CHAPTER XXIX.
54:  CHAPTER XXX.
55:  CHAPTER XXXI.
56:  CHAPTER XXXII.
57:  

In [65]:
lines = raw_texts['mill_on_floss'].split('\n')
for i, line in enumerate(lines[0:100]):
    print(f"{i}: {line}")

0: The Project Gutenberg eBook of The Mill on the Floss, by George Eliot
1: 
2: This eBook is for the use of anyone anywhere in the United States and
3: most other parts of the world at no cost and with almost no restrictions
4: whatsoever. You may copy it, give it away or re-use it under the terms
5: of the Project Gutenberg License included with this eBook or online at
6: www.gutenberg.org. If you are not located in the United States, you
7: will have to check the laws of the country where you are located before
8: using this eBook.
9: 
10: Title: The Mill on the Floss
11: 
12: Author: George Eliot
13: 
14: Release Date: January 13, 2003 [eBook #6688]
15: [Last updated: April 15, 2023]
16: 
17: Language: English
18: 
19: Character set encoding: UTF-8
20: 
21: Produced by: Curtis Weyant and David Maddock
22: 
23: *** START OF THE PROJECT GUTENBERG EBOOK THE MILL ON THE FLOSS ***
24: 
25: cover
26: 
27: 
28: The Mill on the Floss
29: 
30: 
31: 
32: by George Eliot
33: 
34: 
35: 
36: “I

In [66]:
lines = raw_texts['adam_bede'].split('\n')
for i, line in enumerate(lines[0:100]):
    print(f"{i}: {line}")

0: The Project Gutenberg eBook of Adam Bede, by George Eliot
1: 
2: This eBook is for the use of anyone anywhere in the United States and
3: most other parts of the world at no cost and with almost no restrictions
4: whatsoever. You may copy it, give it away or re-use it under the terms
5: of the Project Gutenberg License included with this eBook or online at
6: www.gutenberg.org. If you are not located in the United States, you
7: will have to check the laws of the country where you are located before
8: using this eBook.
9: 
10: Title: Adam Bede
11: 
12: Author: George Eliot [pseudonym of Mary Anne Evans]
13: 
14: Release Date: April, 1996 [eBook #507]
15: [Most recently updated: December 2, 2023]
16: 
17: Language: English
18: 
19: Produced by: An Anonymous Volunteer and David Widger
20: 
21: *** START OF THE PROJECT GUTENBERG EBOOK ADAM BEDE ***
22: 
23: 
24: 
25: 
26: Adam Bede
27: 
28: by George Eliot
29: 
30: 
31: CONTENTS
32: 
33:  Book First
34:  Chapter I — The Workshop
35:  

In [67]:
chap_paths = {
    145: {  # Middlemarch
        'start_line': 100,  # After table of contents
        'end_line': -100,   # Before Gutenberg footer
        'book': re.compile(r'^\s*BOOK\s+[IVXLCDM]+\.', re.IGNORECASE),
        'chapter': re.compile(r'^\s*CHAPTER\s+[IVXLCDM]+\.\s*$')
    },
    6688: {  # Mill on the Floss
        'start_line': 100,
        'end_line': -100,
        'book': re.compile(r'^\s*BOOK\s+\w+\.', re.IGNORECASE),
        'chapter': re.compile(r'^\s*Chapter\s+[IVXLCDM]+\.')  # Note capital C
    },
    507: {  # Adam Bede
        'start_line': 100,
        'end_line': -100,
        'book': re.compile(r'^\s*Book\s+\w+\s*$'),
        'chapter': re.compile(r'^\s*Chapter\s+[IVXLCDM]+\s+—')  # Note em-dash
    }
}

In [68]:
all_chapters = []

for key, info in books.items():
    book_id = info['book_id']
    lines = raw_texts[key].split('\n')
    patterns = chap_paths[book_id]
    
    start = patterns['start_line']
    end = patterns['end_line']
    lines = lines[start:end]
    
    chap_num = 0
    current_chapter = []
    
    for line in lines:
        if patterns['chapter'].match(line):
            # Save previous chapter
            if current_chapter:
                all_chapters.append({
                    'book_id': book_id,
                    'chap_num': chap_num,
                    'chapter_text': '\n'.join(current_chapter)
                })
            chap_num += 1
            current_chapter = []
        else:
            current_chapter.append(line)
    
    if current_chapter:
        all_chapters.append({
            'book_id': book_id,
            'chap_num': chap_num,
            'chapter_text': '\n'.join(current_chapter)
        })
    
    print(f"Parsed {chap_num} chapters from {info['title']} ({len(all_chapters)} total)")

CHAP = pd.DataFrame(all_chapters).set_index(['book_id', 'chap_num'])

Parsed 105 chapters from Middlemarch by George Eliot (88 total)
Parsed 76 chapters from The Mill on the Floss by George Eliot (148 total)
Parsed 0 chapters from Adam Bede by George Eliot (149 total)


In [69]:
doc_list = []

for (book_id, chap_num), row in CHAP.iterrows():
    text = row['chapter_text']
    
    paragraphs = re.split(r'\n\s*\n', text)
    
    for para_num, para_text in enumerate(paragraphs):
        para_text = para_text.strip()
        if para_text:
            para_text = re.sub(r'\n', ' ', para_text)
            
            doc_list.append({
                'book_id': book_id,
                'chap_num': chap_num,
                'para_num': para_num,
                'para_text': para_text
            })

DOC = pd.DataFrame(doc_list).set_index(['book_id', 'chap_num', 'para_num'])
print(f"DOC table created with {len(DOC)} paragraphs")

DOC table created with 10392 paragraphs


In [70]:
token_list = []

for (book_id, chap_num, para_num), row in DOC.iterrows():
    para_text = row['para_text']
    
    sentences = nltk.sent_tokenize(para_text)
    
    for sent_num, sentence in enumerate(sentences):
        tokens = nltk.word_tokenize(sentence)
        
        pos_tags = nltk.pos_tag(tokens)
        
        for token_num, (token, pos) in enumerate(pos_tags):
            token_list.append({
                'book_id': book_id,
                'chap_num': chap_num,
                'para_num': para_num,
                'sent_num': sent_num,
                'token_num': token_num,
                'token_str': token,
                'pos': pos
            })
    
    if para_num % 100 == 0:
        print(f"  Processed book {book_id}, chapter {chap_num}, paragraph {para_num}")

TOKEN = pd.DataFrame(token_list)

TOKEN['term_str'] = TOKEN['token_str'].str.lower()
term_to_id = {term: idx for idx, term in enumerate(TOKEN['term_str'].unique())}
TOKEN['term_id'] = TOKEN['term_str'].map(term_to_id)

TOKEN = TOKEN.set_index(OHCO)

print(f"TOKEN table created with {len(TOKEN)} tokens")

  Processed book 145, chapter 4, paragraph 0
  Processed book 145, chapter 19, paragraph 0
  Processed book 145, chapter 20, paragraph 0
  Processed book 145, chapter 21, paragraph 0
  Processed book 145, chapter 22, paragraph 0
  Processed book 145, chapter 23, paragraph 0
  Processed book 145, chapter 24, paragraph 0
  Processed book 145, chapter 25, paragraph 0
  Processed book 145, chapter 26, paragraph 0
  Processed book 145, chapter 27, paragraph 0
  Processed book 145, chapter 28, paragraph 0
  Processed book 145, chapter 29, paragraph 0
  Processed book 145, chapter 30, paragraph 0
  Processed book 145, chapter 31, paragraph 0
  Processed book 145, chapter 31, paragraph 100
  Processed book 145, chapter 32, paragraph 0
  Processed book 145, chapter 33, paragraph 0
  Processed book 145, chapter 34, paragraph 0
  Processed book 145, chapter 35, paragraph 0
  Processed book 145, chapter 36, paragraph 0
  Processed book 145, chapter 37, paragraph 0
  Processed book 145, chapter 38,

In [71]:
VOCAB = TOKEN.groupby('term_id').agg(
    term_str=('term_str', 'first'),
    n=('term_str', 'count')
).reset_index().set_index('term_id')

print(f"VOCAB has {len(VOCAB)} unique terms")

VOCAB has 28275 unique terms


In [72]:
stop_words = set(stopwords.words('english'))
VOCAB['stop'] = VOCAB['term_str'].apply(lambda x: 1 if x in stop_words else 0)

print(f"{VOCAB['stop'].sum()} stopwords identified")

152 stopwords identified


In [73]:
stemmer = PorterStemmer()
VOCAB['p_stem'] = VOCAB['term_str'].apply(stemmer.stem)

In [74]:
pos_counts = TOKEN.reset_index().groupby(['term_id', 'pos']).size().reset_index(name='count')

idx = pos_counts.groupby('term_id')['count'].idxmax()
pos_max_df = pos_counts.loc[idx, ['term_id', 'pos']].set_index('term_id')
pos_max_df = pos_max_df.rename(columns={'pos': 'pos_max'})

if 'pos_max' in VOCAB.columns:
    VOCAB['pos_max'] = pos_max_df['pos_max']
else:
    VOCAB = VOCAB.join(pos_max_df)

VOCAB.head()


,term_str,n,stop,p_stem,pos_max
term_id,,,,,
0,book,141,0,book,NN
1,viii,3,0,viii,NNP
2,.,27936,0,.,.
3,sunset,8,0,sunset,NN
4,and,21261,1,and,CC


In [75]:
LIB.to_csv('LIBRARY.csv')
DOC.to_csv('DOC.csv')
TOKEN.to_csv('TOKEN.csv')
VOCAB.to_csv('VOCAB.csv')

print(f"LIBRARY: {len(LIB)} books")
print(f"DOC: {len(DOC)} paragraphs")
print(f"TOKEN: {len(TOKEN)} tokens")
print(f"VOCAB: {len(VOCAB)} unique terms")

LIBRARY: 3 books
DOC: 10392 paragraphs
TOKEN: 890366 tokens
VOCAB: 28275 unique terms
